## Imports

In [21]:
import random
from pathlib import Path
import json
import re

## Configuration

In [22]:
# Set random seed for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# Split ratios
TRAIN_RATIO = 0.8
DEV_RATIO = 0.1
TEST_RATIO = 0.1

# Language pairs to split
language_pairs = [
    "en-war",  # baseline + distant
    "en-ceb",  # similar donor
]

print(f"Random seed: {RANDOM_SEED}")
print(f"Split ratios: Train={TRAIN_RATIO*100}%, Dev={DEV_RATIO*100}%, Test={TEST_RATIO*100}%")
print(f"Total language pairs: {len(language_pairs)}")

Random seed: 42
Split ratios: Train=80.0%, Dev=10.0%, Test=10.0%
Total language pairs: 2


## Load Parallel Corpora

In [23]:
parallel_dir = Path("../data/parallel")

# Load corpus data for each pair
corpus_data = {}

for pair in language_pairs:
    src_code, tgt_code = pair.split("-")
    pair_dir = parallel_dir / pair
    
    # Read source file
    src_file = pair_dir / f"{pair}.{src_code}"
    with open(src_file, "r", encoding="utf-8") as f:
        src_lines = [line.strip() for line in f.readlines()]
    
    # Read target file
    tgt_file = pair_dir / f"{pair}.{tgt_code}"
    with open(tgt_file, "r", encoding="utf-8") as f:
        tgt_lines = [line.strip() for line in f.readlines()]
    
    # Store as parallel sentence pairs
    corpus_data[pair] = {
        "src": src_lines,
        "tgt": tgt_lines,
        "src_code": src_code,
        "tgt_code": tgt_code,
        "total": len(src_lines)
    }
    
    print(f"Loaded {pair}: {len(src_lines):,} parallel sentences")

print(f"\n✓ All {len(corpus_data)} language pairs loaded")

Loaded en-war: 9,272 parallel sentences
Loaded en-ceb: 9,272 parallel sentences

✓ All 2 language pairs loaded


## Clean Training Data

In [24]:
# Common proper nouns in the Bible (English and their equivalents)
# These should always be capitalized
PROPER_NOUNS = {
    # single I shoyld always be capitalized
    "i ",
    # Deity and Divine Names
    "god", "lord", "jehovah", "yahweh", "elohim", "adonai", "almighty",
    "holy spirit", "messiah", "christ", "emmanuel",
    
    # Major Biblical Figures - Old Testament
    "adam", "eve", "cain", "abel", "seth", "enoch", "noah", "shem", "ham", "japheth",
    "abraham", "sarah", "isaac", "rebekah", "jacob", "rachel", "leah", "esau",
    "joseph", "reuben", "simeon", "levi", "judah", "dan", "naphtali", "gad",
    "asher", "issachar", "zebulun", "benjamin", "dinah",
    "moses", "aaron", "miriam", "joshua", "caleb", "rahab", "deborah", "gideon",
    "samson", "delilah", "ruth", "naomi", "boaz", "hannah", "samuel", "eli",
    "saul", "david", "jonathan", "bathsheba", "solomon", "rehoboam", "jeroboam",
    "elijah", "elisha", "isaiah", "jeremiah", "ezekiel", "daniel", "hosea",
    "joel", "amos", "obadiah", "jonah", "micah", "nahum", "habakkuk",
    "zephaniah", "haggai", "zechariah", "malachi", "job", "esther", "mordecai",
    "ezra", "nehemiah", "haman", "goliath", "absalom", "jezebel", "ahab", "rameses",
    
    
    # Major Biblical Figures - New Testament
    "jesus", "mary", "john", "peter", "andrew", "james", "philip",
    "bartholomew", "matthew", "thomas", "thaddaeus", "simon", "judas", "matthias",
    "paul", "barnabas", "timothy", "titus", "luke", "mark", "stephen",
    "priscilla", "aquila", "apollos", "silas", "martha", "lazarus", "nicodemus",
    "zacchaeus", "mary magdalene", "elizabeth", "john baptist",
    "herod", "pilate", "caiaphas", "annas", "barabbas", "pontius pilate", "lucifer",
    "lucius", "demas", "epaphras", "epaphroditus", "philemon", "onesimus", "titus",
    "silvanus",
    
    # Places - Regions and Countries
    "eden", "egypt", "canaan", "israel", "judah", "judea", "samaria", "galilee",
    "syria", "assyria", "babylon", "babylonia", "persia", "media", "greece",
    "macedonia", "rome", "phrygia", "galatia", "cappadocia", "pontus",
    "bithynia", "cilicia", "pamphylia", "lycia", "crete", "cyprus", "malta",
    "arabia", "ethiopia", "libya", "mesopotamia", "chaldea", "elam", "moab",
    "edom", "ammon", "philistia", "phoenicia", "tyre", "sidon", "cyrene",
    "damascus", "tarsus", "corinth", "athens", "ephesus", "philippi",
    "thessalonica", "berea", "smyrna", "pergamum", "thyatira", "sardis",
    "philadelphia", "laodicea", "colosse", "troas", "miletus", "patmos",

    
    # Cities and Towns
    "jerusalem", "bethlehem", "nazareth", "capernaum", "bethany", "jericho",
    "hebron", "beersheba", "shechem", "dothan", "megiddo", "jezreel",
    "ramah", "shiloh", "gilgal", "mizpah", "gibeon", "joppa", "caesarea",
    "antioch", "damascus", "tarsus", "corinth", "athens", "ephesus", "philippi",
    "thessalonica", "berea", "smyrna", "pergamum", "thyatira", "sardis",
    "philadelphia", "laodicea", "colosse", "troas", "miletus", "patmos",
    "gaza", "gath", "ashkelon", "ekron", "ashdod",
    
    # Geographic Features
    "jordan", "euphrates", "tigris", "nile", "mediterranean", "red sea",
    "dead sea", "sea galilee", "sinai", "horeb", "ararat", "moriah",
    "carmel", "zion", "olivet", "tabor", "hermon", "gilead", "bashan",
    "sharon", "negev",
    
    # Tribes and Peoples
    "israelites", "hebrews", "jews", "levites", "pharisees", "sadducees",
    "scribes", "zealots", "samaritans", "gentiles", "romans", "greeks",
    "egyptians", "philistines", "canaanites", "amorites", "hittites", "perizzites",
    "hivites", "jebusites", "moabites", "edomites", "ammonites", "midianites",
    "amalekites", "assyrians", "babylonians", "persians", "medes", "chaldeans",
    "jewish"
    
    # Important Structures (when referring to THE specific structure)
    "solomon temple", "ark covenant",
    
    # Tagalog equivalents
    "diyos", "panginoon", "hesus", "kristo", "moises", "abraham", "isaac", "jacob",
    "jose", "david", "solomon", "samuel", "pablo", "pedro", "juan", "santiago",
    "maria", "marta", "lazaro", "faraon", "ehipto", "israel", "jerusalem",
    "galilea", "huda", "hudas", "benjamin", "levita", "nazaret", "belen",
    "jordan", "mediterraneo", "babilonia", "asiria", "persia", "roma",
    "espiritu santo", "makapangyarihan", "mesiyas", "emmanuel",
    "goliath", "absalom", "jezebel", "ahab", "rameses",
    
    # Cebuano equivalents
    "dios", "ginoo", "jesus", "cristo", "moises", "abraham", "isaac", "jacob",
    "jose", "david", "solomon", "samuel", "pablo", "pedro", "juan", "santiago",
    "maria", "marta", "lazaro", "faraon", "ehipto", "israel", "jerusalem",
    "galilea", "juda", "judas", "benjamin", "levita", "nazaret", "belen",
    "jordan", "mediterraneo", "babilonia", "asiria", "persia", "roma",
    "espiritu santo", "makagagahum", "mesiyas", "emanuel",
    "goliath", "absalom", "jezebel", "ahab", "rameses",
    
    # Waray equivalents
    "diyos", "ginoo", "hesus", "kristo", "moises", "abraham", "isaac", "jacob",
    "jose", "david", "solomon", "samuel", "pablo", "pedro", "juan", "santiago",
    "maria", "marta", "lazaro", "faraon", "ehipto", "israel", "jerusalem",
    "galilea", "juda", "hudas", "benjamin", "levita", "nazaret", "belen",
    "jordan", "mediterraneo", "babilonia", "asiria", "persia", "roma",
    "espiritu santo", "makagagahom", "mesiyas", "emanuel",
    "goliath", "absalom", "jezebel", "ahab", "rameses"
}

# Words that should always be lowercase 
COMMON_ARTICLES = {
    "the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for",
    "of", "with", "by", "from", "as", "is", "was", "are", "were", "be",
    "ang", "ng", "sa", "ay", "mga", "si", "ni", "kay" 
}

def apply_true_casing(text):
    """
    Apply true casing to text:
    - Capitalize proper nouns (biblical names, places, deity references)
    - Lowercase common nouns and articles
    - Keep first word of sentence capitalized
    - Handle multi-word proper nouns (e.g., "Holy Spirit")
    """
    if not text:
        return text
    
    # First, lowercase everything to start fresh
    text_lower = text.lower()
    
    # Split into words while preserving spaces
    words = text_lower.split()
    result_words = []
    
    for i, word in enumerate(words):
        # Remove punctuation for comparison but keep it for the result
        word_clean = re.sub(r'[^\w\s]', '', word)
        word_clean_lower = word_clean.lower()
        
        # Get leading/trailing punctuation
        leading_punct = re.match(r'^([^\w]*)', word).group(1)
        trailing_punct = re.search(r'([^\w]*)$', word).group(1)
        
        # Check if it's a proper noun
        is_proper = False
        
        # Check single word proper nouns
        if word_clean_lower in PROPER_NOUNS:
            is_proper = True
            cased_word = word_clean.capitalize()
        
        # Check multi-word proper nouns (look ahead)
        elif i < len(words) - 1:
            next_word = re.sub(r'[^\w\s]', '', words[i + 1]).lower()
            bigram = f"{word_clean_lower} {next_word}"
            if bigram in PROPER_NOUNS:
                is_proper = True
                # Capitalize both words in the bigram
                cased_word = word_clean.capitalize()
                next_cased = re.sub(r'[^\w\s]', '', words[i + 1]).capitalize()
                next_leading = re.match(r'^([^\w]*)', words[i + 1]).group(1)
                next_trailing = re.search(r'([^\w]*)$', words[i + 1]).group(1)
                result_word = leading_punct + cased_word + trailing_punct
                result_next = next_leading + next_cased + next_trailing
                result_words.append(result_word)
                result_words.append(result_next)
                # Skip next word in the loop
                i += 1
                continue
            else:
                cased_word = word_clean 
        
        # First word of sentence should be capitalized ONLY if it is a proper noun
        elif i == 0 and word_clean_lower in PROPER_NOUNS:
            is_proper = True
            cased_word = word_clean.capitalize()
        
        # Check if previous word ends with sentence-ending punctuation
        elif i > 0 and any(p in result_words[-1] for p in ['.', '!', '?']):
            cased_word = word_clean.capitalize()
            is_proper = True
        
        else:
            # Keep lowercase for common words
            cased_word = word_clean
        
        # Reconstruct with punctuation
        result_word = leading_punct + cased_word + trailing_punct
        result_words.append(result_word)
    
    return ' '.join(result_words)

 # Apply true casing
    text = apply_true_casing(text)
    

## Create Deterministic Split Indices

In [25]:
# Get total number of sentences (should be same for all pairs)
total_sentences = corpus_data[language_pairs[0]]["total"]

# Verify all pairs have same size
for pair in language_pairs:
    assert corpus_data[pair]["total"] == total_sentences, f"Size mismatch for {pair}"

print(f"Total sentences per corpus: {total_sentences:,}")

# Create shuffled indices
indices = list(range(total_sentences))
random.shuffle(indices)

# Calculate split sizes
train_size = int(total_sentences * TRAIN_RATIO)
dev_size = int(total_sentences * DEV_RATIO)
test_size = total_sentences - train_size - dev_size  # remaining goes to test

# Split indices
train_indices = indices[:train_size]
dev_indices = indices[train_size:train_size + dev_size]
test_indices = indices[train_size + dev_size:]

print(f"\nSplit sizes:")
print(f"  Train: {len(train_indices):,} sentences ({len(train_indices)/total_sentences*100:.1f}%)")
print(f"  Dev:   {len(dev_indices):,} sentences ({len(dev_indices)/total_sentences*100:.1f}%)")
print(f"  Test:  {len(test_indices):,} sentences ({len(test_indices)/total_sentences*100:.1f}%)")
print(f"  Total: {len(train_indices) + len(dev_indices) + len(test_indices):,} sentences")

Total sentences per corpus: 9,272

Split sizes:
  Train: 7,417 sentences (80.0%)
  Dev:   927 sentences (10.0%)
  Test:  928 sentences (10.0%)
  Total: 9,272 sentences


## Split and Save Data

In [26]:
def save_split(src_lines, tgt_lines, indices, output_dir, split_name, src_code, tgt_code, pair_name):
    """
    Save a data split (train/dev/test) to files.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Extract sentences at specified indices
    src_split = [src_lines[i] for i in indices]
    tgt_split = [tgt_lines[i] for i in indices]
    
    # Save source file
    src_file = output_dir / f"{split_name}.{src_code}"
    with open(src_file, "w", encoding="utf-8") as f:
        for line in src_split:
            f.write(f"{line}\n")
    
    # Save target file
    tgt_file = output_dir / f"{split_name}.{tgt_code}"
    with open(tgt_file, "w", encoding="utf-8") as f:
        for line in tgt_split:
            f.write(f"{line}\n")
    
    return len(src_split)

# Create splits directory
splits_dir = Path("../data/splits")
splits_dir.mkdir(parents=True, exist_ok=True)

print("Creating splits for all language pairs...")
print("=" * 80)

split_stats = []

for pair in language_pairs:
    print(f"\nProcessing {pair}...")
    
    data = corpus_data[pair]
    pair_dir = splits_dir / pair
    
    # Save train split
    train_count = save_split(
        data["src"], data["tgt"], train_indices, 
        pair_dir, "train", data["src_code"], data["tgt_code"], pair
    )
    
    # Save dev split
    dev_count = save_split(
        data["src"], data["tgt"], dev_indices, 
        pair_dir, "dev", data["src_code"], data["tgt_code"], pair
    )
    
    # Save test split
    test_count = save_split(
        data["src"], data["tgt"], test_indices, 
        pair_dir, "test", data["src_code"], data["tgt_code"], pair
    )
    
    split_stats.append({
        "pair": pair,
        "train": train_count,
        "dev": dev_count,
        "test": test_count,
        "total": train_count + dev_count + test_count
    })
    
    print(f"  ✓ Train: {train_count:,} | Dev: {dev_count:,} | Test: {test_count:,}")

print("\n" + "=" * 80)
print("✓ All splits created successfully!")

Creating splits for all language pairs...

Processing en-war...
  ✓ Train: 7,417 | Dev: 927 | Test: 928

Processing en-ceb...
  ✓ Train: 7,417 | Dev: 927 | Test: 928

✓ All splits created successfully!


## Normalize Casing of Training Data

In [27]:
# Common proper nouns in the Bible (English and their equivalents)
# These should always be capitalized
PROPER_NOUNS = {
    # single I shoyld always be capitalized 
    "i",
    # Deity and Divine Names
    "god", "lord", "jehovah", "yahweh", "elohim", "adonai", "almighty",
    "holy spirit", "messiah", "christ", "emmanuel",
    
    # Major Biblical Figures - Old Testament
    "adam", "eve", "cain", "abel", "seth", "enoch", "noah", "shem", "ham", "japheth",
    "abraham", "sarah", "isaac", "rebekah", "jacob", "rachel", "leah", "esau",
    "joseph", "reuben", "simeon", "levi", "judah", "dan", "naphtali", "gad",
    "asher", "issachar", "zebulun", "benjamin", "dinah",
    "moses", "aaron", "miriam", "joshua", "caleb", "rahab", "deborah", "gideon",
    "samson", "delilah", "ruth", "naomi", "boaz", "hannah", "samuel", "eli",
    "saul", "david", "jonathan", "bathsheba", "solomon", "rehoboam", "jeroboam",
    "elijah", "elisha", "isaiah", "jeremiah", "ezekiel", "daniel", "hosea",
    "joel", "amos", "obadiah", "jonah", "micah", "nahum", "habakkuk",
    "zephaniah", "haggai", "zechariah", "malachi", "job", "esther", "mordecai",
    "ezra", "nehemiah", "haman", "goliath", "absalom", "jezebel", "ahab", "rameses",
    
    
    # Major Biblical Figures - New Testament
    "jesus", "mary", "john", "peter", "andrew", "james", "philip",
    "bartholomew", "matthew", "thomas", "thaddaeus", "simon", "judas", "matthias",
    "paul", "barnabas", "timothy", "titus", "luke", "mark", "stephen",
    "priscilla", "aquila", "apollos", "silas", "martha", "lazarus", "nicodemus",
    "zacchaeus", "mary magdalene", "elizabeth", "john baptist",
    "herod", "pilate", "caiaphas", "annas", "barabbas", "pontius pilate", "lucifer",
    "lucius", "demas", "epaphras", "epaphroditus", "philemon", "onesimus", "titus",
    "silvanus",
    
    # Places - Regions and Countries
    "eden", "egypt", "canaan", "israel", "judah", "judea", "samaria", "galilee",
    "syria", "assyria", "babylon", "babylonia", "persia", "media", "greece",
    "macedonia", "rome", "phrygia", "galatia", "cappadocia", "pontus",
    "bithynia", "cilicia", "pamphylia", "lycia", "crete", "cyprus", "malta",
    "arabia", "ethiopia", "libya", "mesopotamia", "chaldea", "elam", "moab",
    "edom", "ammon", "philistia", "phoenicia", "tyre", "sidon", "cyrene",
    "damascus", "tarsus", "corinth", "athens", "ephesus", "philippi",
    "thessalonica", "berea", "smyrna", "pergamum", "thyatira", "sardis",
    "philadelphia", "laodicea", "colosse", "troas", "miletus", "patmos",

    
    # Cities and Towns
    "jerusalem", "bethlehem", "nazareth", "capernaum", "bethany", "jericho",
    "hebron", "beersheba", "shechem", "dothan", "megiddo", "jezreel",
    "ramah", "shiloh", "gilgal", "mizpah", "gibeon", "joppa", "caesarea",
    "antioch", "damascus", "tarsus", "corinth", "athens", "ephesus", "philippi",
    "thessalonica", "berea", "smyrna", "pergamum", "thyatira", "sardis",
    "philadelphia", "laodicea", "colosse", "troas", "miletus", "patmos",
    "gaza", "gath", "ashkelon", "ekron", "ashdod",
    
    # Geographic Features
    "jordan", "euphrates", "tigris", "nile", "mediterranean", "red sea",
    "dead sea", "sea galilee", "sinai", "horeb", "ararat", "moriah",
    "carmel", "zion", "olivet", "tabor", "hermon", "gilead", "bashan",
    "sharon", "negev",
    
    # Tribes and Peoples
    "israelites", "hebrews", "jews", "levites", "pharisees", "sadducees",
    "scribes", "zealots", "samaritans", "gentiles", "romans", "greeks",
    "egyptians", "philistines", "canaanites", "amorites", "hittites", "perizzites",
    "hivites", "jebusites", "moabites", "edomites", "ammonites", "midianites",
    "amalekites", "assyrians", "babylonians", "persians", "medes", "chaldeans",
    "jewish"
    
    # Important Structures (when referring to THE specific structure)
    "solomon temple", "ark covenant",
    
    # Tagalog equivalents
    "diyos", "panginoon", "hesus", "kristo", "moises", "abraham", "isaac", "jacob",
    "jose", "david", "solomon", "samuel", "pablo", "pedro", "juan", "santiago",
    "maria", "marta", "lazaro", "faraon", "ehipto", "israel", "jerusalem",
    "galilea", "huda", "hudas", "benjamin", "levita", "nazaret", "belen",
    "jordan", "mediterraneo", "babilonia", "asiria", "persia", "roma",
    "espiritu santo", "makapangyarihan", "mesiyas", "emmanuel",
    "goliath", "absalom", "jezebel", "ahab", "rameses",
    
    # Cebuano equivalents
    "dios", "ginoo", "jesus", "cristo", "moises", "abraham", "isaac", "jacob",
    "jose", "david", "solomon", "samuel", "pablo", "pedro", "juan", "santiago",
    "maria", "marta", "lazaro", "faraon", "ehipto", "israel", "jerusalem",
    "galilea", "juda", "judas", "benjamin", "levita", "nazaret", "belen",
    "jordan", "mediterraneo", "babilonia", "asiria", "persia", "roma",
    "espiritu santo", "makagagahum", "mesiyas", "emanuel",
    "goliath", "absalom", "jezebel", "ahab", "rameses",
    
    # Waray equivalents
    "diyos", "ginoo", "hesus", "kristo", "moises", "abraham", "isaac", "jacob",
    "jose", "david", "solomon", "samuel", "pablo", "pedro", "juan", "santiago",
    "maria", "marta", "lazaro", "faraon", "ehipto", "israel", "jerusalem",
    "galilea", "juda", "hudas", "benjamin", "levita", "nazaret", "belen",
    "jordan", "mediterraneo", "babilonia", "asiria", "persia", "roma",
    "espiritu santo", "makagagahom", "mesiyas", "emanuel",
    "goliath", "absalom", "jezebel", "ahab", "rameses"
}

# Words that should always be lowercase 
COMMON_ARTICLES = {
    "the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for",
    "of", "with", "by", "from", "as", "is", "was", "are", "were", "be",
    "ang", "ng", "sa", "ay", "mga", "si", "ni", "kay" 
}

def apply_true_casing(text):
    """
    Apply true casing to text:
    - Capitalize proper nouns (biblical names, places, deity references)
    - Lowercase common nouns and articles
    - Keep first word of sentence capitalized
    - Handle multi-word proper nouns (e.g., "Holy Spirit")
    """
    if not text:
        return text
    
    # First, lowercase everything to start fresh
    text_lower = text.lower()
    
    # Split into words while preserving spaces
    words = text_lower.split()
    result_words = []
    
    for i, word in enumerate(words):
        # Remove punctuation for comparison but keep it for the result
        word_clean = re.sub(r'[^\w\s]', '', word)
        word_clean_lower = word_clean.lower()
        
        # Get leading/trailing punctuation
        leading_punct = re.match(r'^([^\w]*)', word).group(1)
        trailing_punct = re.search(r'([^\w]*)$', word).group(1)
        
        # Check if it's a proper noun
        is_proper = False
        
        # Check single word proper nouns
        if word_clean_lower in PROPER_NOUNS:
            is_proper = True
            cased_word = word_clean.capitalize()
        
        # Check multi-word proper nouns (look ahead)
        elif i < len(words) - 1:
            next_word = re.sub(r'[^\w\s]', '', words[i + 1]).lower()
            bigram = f"{word_clean_lower} {next_word}"
            if bigram in PROPER_NOUNS:
                is_proper = True
                # Capitalize both words in the bigram
                cased_word = word_clean.capitalize()
                next_cased = re.sub(r'[^\w\s]', '', words[i + 1]).capitalize()
                next_leading = re.match(r'^([^\w]*)', words[i + 1]).group(1)
                next_trailing = re.search(r'([^\w]*)$', words[i + 1]).group(1)
                result_word = leading_punct + cased_word + trailing_punct
                result_next = next_leading + next_cased + next_trailing
                result_words.append(result_word)
                result_words.append(result_next)
                # Skip next word in the loop
                i += 1
                continue
            else:
                cased_word = word_clean 
        
        # First word of sentence should be capitalized ONLY if it is a proper noun
        elif i == 0 and word_clean_lower in PROPER_NOUNS:
            is_proper = True
            cased_word = word_clean.capitalize()
        
        # Check if previous word ends with sentence-ending punctuation
        elif i > 0 and any(p in result_words[-1] for p in ['.', '!', '?']):
            cased_word = word_clean.capitalize()
            is_proper = True
        
        else:
            # Keep lowercase for common words
            cased_word = word_clean
        
        # Reconstruct with punctuation
        result_word = leading_punct + cased_word + trailing_punct
        result_words.append(result_word)
    
    return ' '.join(result_words)

 # Apply true casing
    text = apply_true_casing(text)
    

In [28]:
# Clean only the training split for casing
for pair in language_pairs:
    print(f"\nCleaning casing for training split: {pair}")
    pair_dir = splits_dir / pair
    src_code, tgt_code = pair.split("-")
    train_src_file = pair_dir / f"train.{src_code}"
    train_tgt_file = pair_dir / f"train.{tgt_code}"

    # Read train split
    with open(train_src_file, "r", encoding="utf-8") as f:
        train_src_lines = [line.strip() for line in f.readlines()]
    with open(train_tgt_file, "r", encoding="utf-8") as f:
        train_tgt_lines = [line.strip() for line in f.readlines()]

    # Clean casing
    train_src_cleaned = [apply_true_casing(line) for line in train_src_lines]
    train_tgt_cleaned = [apply_true_casing(line) for line in train_tgt_lines]

    # Overwrite train split with cleaned casing
    with open(train_src_file, "w", encoding="utf-8") as f:
        for line in train_src_cleaned:
            f.write(f"{line}\n")
    with open(train_tgt_file, "w", encoding="utf-8") as f:
        for line in train_tgt_cleaned:
            f.write(f"{line}\n")

    print(f"✓ Cleaned casing for {pair} training split")


Cleaning casing for training split: en-war
✓ Cleaned casing for en-war training split

Cleaning casing for training split: en-ceb
✓ Cleaned casing for en-ceb training split


## Save Split Metadata

In [29]:
# Save split metadata
split_metadata = {
    "description": "Train/Dev/Test splits for MT training",
    "random_seed": RANDOM_SEED,
    "split_ratios": {
        "train": TRAIN_RATIO,
        "dev": DEV_RATIO,
        "test": TEST_RATIO
    },
    "total_sentences": total_sentences,
    "split_sizes": {
        "train": len(train_indices),
        "dev": len(dev_indices),
        "test": len(test_indices)
    },
    "language_pairs": split_stats
}

metadata_file = splits_dir / "split_metadata.json"
with open(metadata_file, "w", encoding="utf-8") as f:
    json.dump(split_metadata, f, indent=2)

print(f"✓ Metadata saved to {metadata_file}")

# Save indices for reproducibility
indices_data = {
    "random_seed": RANDOM_SEED,
    "train_indices": train_indices,
    "dev_indices": dev_indices,
    "test_indices": test_indices
}

indices_file = splits_dir / "split_indices.json"
with open(indices_file, "w", encoding="utf-8") as f:
    json.dump(indices_data, f, indent=2)

print(f"✓ Split indices saved to {indices_file}")

✓ Metadata saved to ..\data\splits\split_metadata.json
✓ Split indices saved to ..\data\splits\split_indices.json


## Verify Splits

In [30]:
# Verify no overlap between splits
train_set = set(train_indices)
dev_set = set(dev_indices)
test_set = set(test_indices)

assert len(train_set & dev_set) == 0, "Train and dev overlap!"
assert len(train_set & test_set) == 0, "Train and test overlap!"
assert len(dev_set & test_set) == 0, "Dev and test overlap!"

print("✓ No overlap between splits - verified!")

# Show sample from a split
sample_pair = "en-war"
pair_dir = splits_dir / sample_pair
src_code, tgt_code = sample_pair.split("-")

# Read first 3 sentences from train split
train_src = pair_dir / f"train.{src_code}"
train_tgt = pair_dir / f"train.{tgt_code}"

with open(train_src, "r", encoding="utf-8") as f:
    src_samples = [line.strip() for line in f.readlines()[:3]]

with open(train_tgt, "r", encoding="utf-8") as f:
    tgt_samples = [line.strip() for line in f.readlines()[:3]]

print(f"\nSample from {sample_pair.upper()} train split:")
print("=" * 100)
for i, (src, tgt) in enumerate(zip(src_samples, tgt_samples), 1):
    print(f"\n{i}. {src_code.upper()}: {src}")
    print(f"   {tgt_code.upper()}: {tgt}")
print("=" * 100)

✓ No overlap between splits - verified!

Sample from EN-WAR train split:

1. EN: he alone stretches out the heavens and treads on the waves of the sea.
   WAR: waray may binmulig han dyos hin pagbitad han kalangitan, o hin pagtamak han taludtod han higante nga mananap ha dagat.

2. EN: he rescued me from my powerful enemy, from my foes, who were too strong for me.
   WAR: tikang han mga kaaway gintalwas ako han mga nagdudumot nga mga tawo. waray ako makaato, kay makusog hira hin duro.

3. EN: that is why it has been called the field of blood to this day.
   WAR: sanglit ginngaranan an uma nga “uma hin dugo” ngada yana nga adlaw.
